# Phase 34 — v1.3 First Measured Computational Efficiency / Runtime Profiling
## Reviewer Notebook

**Classification:** `READ_ONLY_PHASE34_V1_3_MEASURED_EFFICIENCY_RUNTIME_PROFILING_REVIEWER_NOTEBOOK`
**Revision:** `ra_medfilm_barr_phase34_efficiency_profiling_v1_3`

Frozen v1.3 harness + worker were used with canonical component semantics. Evidence classes are
kept separate: historical analytical compute · static model resources · measured
hardware-specific runtime · conditional computation.


## 0. Setup

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path(r"D:\Softfors\jup")
AUD = ROOT / "reports" / "audits"
TAB = ROOT / "reports" / "tables"


def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def tshow(name, n=None):
    df = pd.read_csv(TAB / name)
    print(name, "| shape:", df.shape)
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None, "display.width", 260):
        print(df.to_string(index=False))
    return df


print("Phase34 v1.3 measured efficiency / runtime profiling review")
print("READ-ONLY. No checkpoint load, no model construction, no forward, no timing rerun,")
print("no memory measurement, no profiler, no protected data, no TEST, no training.")


Phase34 v1.3 measured efficiency / runtime profiling review
READ-ONLY. No checkpoint load, no model construction, no forward, no timing rerun,
no memory measurement, no profiler, no protected data, no TEST, no training.


## 1. Authority, source freeze and gates

In [2]:
pf = jload(AUD / "phase34_execution_source_preflight_measured_v1_3.json")
print("additive preflight:", pf["additive_filename"],
      "| all sources match:", pf["all_sources_match"],
      "| smoke driver used:", pf["smoke_only_driver_used_for_measurement"])
sd = jload(AUD / "phase34_smoke_only_driver_scope_record_v1_3.json")
print("smoke driver role:", sd["driver_role"],
      "| eligible for measured execution:", sd["eligible_for_measured_execution"])
env = jload(AUD / "phase34_measured_execution_environment_v1_3.json")
print("environment gate:", env["environment_gate_pass"])
for k, v in env["environment_checks"].items():
    print(f"  {k:28s} = {v}")


additive preflight: True | all sources match: True | smoke driver used: False
smoke driver role: SMOKE_ONLY_ORCHESTRATION_AND_REPORTING | eligible for measured execution: False
environment gate: True
  gpu_model                    = True
  gpu_compute_capability       = True
  gpu_total_vram_bytes         = True
  python_version               = True
  torch_version                = True
  cuda_runtime                 = True
  cuda_driver_version          = True
  cudnn_version                = True


## 2. Component / checkpoint identity and parameters

In [3]:
tshow("phase34_static_resource_profile_v1_3.csv")

phase34_static_resource_profile_v1_3.csv | shape: (4, 8)
component  parameter_count  trainable_parameters  non_trainable_parameters  parameter_bytes_float32  checkpoint_bytes                                                checkpoint_sha256             execution_role
        G          3523001               3523001                         0                 14092004          42369710 286689fdd5c63ad75a383151d267af8b476e7a98327632de292ce6ed5b5ba1b7            always executed
    R_psi          1276401               1276401                         0                  5105604          15425424 6d28608a669df497740b6594ab346136f4efdf3f66002a868a1f1a3ec6689109        EXECUTE branch only
    B_phi            36801                 36801                         0                   147204            455682 8a412b2fe82f225f844fda07379a6436c1d73d435f885b26a1d88d5b2f9fc43b pre-gate (always executed)
  S_omega             1217                  1217                         0                     4868    

,component,parameter_count,trainable_parameters,non_trainable_parameters,parameter_bytes_float32,checkpoint_bytes,checkpoint_sha256,execution_role
0,G,3523001,3523001,0,14092004,42369710,286689fdd5c63ad75a383151d267af8b476e7a98327632...,always executed
1,R_psi,1276401,1276401,0,5105604,15425424,6d28608a669df497740b6594ab346136f4efdf3f66002a...,EXECUTE branch only
2,B_phi,36801,36801,0,147204,455682,8a412b2fe82f225f844fda07379a6436c1d73d435f885b...,pre-gate (always executed)
3,S_omega,1217,1217,0,4868,7816,eee9b97b9921e53fc07eac9281dce64ba518c6e1659916...,EXECUTE branch only


## 3. Historical analytical MAC comparators

In [4]:
tshow("phase34_analytical_compute_summary_v1_3.csv")

phase34_analytical_compute_summary_v1_3.csv | shape: (9, 7)
                       reference                                                formula         macs router_present refiner_present s_omega_present                      label
          LITERAL_GLOBAL_ONLY_C1                                                 MAC(G) 3.155174e+10          False           False           False ANALYTICAL_MAC_LOWER_BOUND
        LITERAL_ALWAYS_REFINE_C2                                      MAC(G)+MAC(R_psi) 3.478240e+10          False            True           False ANALYTICAL_MAC_LOWER_BOUND
FINAL_A8_C9_EXPECTED_CONDITIONAL MAC(G)+MAC(B_phi)+p_activate*(MAC(R_psi)+MAC(S_omega)) 3.243790e+10           True            True            True ANALYTICAL_MAC_LOWER_BOUND
            GATED_EXECUTE_ALL_C9              MAC(G)+MAC(B_phi)+MAC(R_psi)+MAC(S_omega) 3.478244e+10           True            True            True ANALYTICAL_MAC_LOWER_BOUND
                RATIO_A8C9_vs_C1                                 

,reference,formula,macs,router_present,refiner_present,s_omega_present,label
0,LITERAL_GLOBAL_ONLY_C1,MAC(G),3.155174e+10,False,False,False,ANALYTICAL_MAC_LOWER_BOUND
1,LITERAL_ALWAYS_REFINE_C2,MAC(G)+MAC(R_psi),3.478240e+10,False,True,False,ANALYTICAL_MAC_LOWER_BOUND
2,FINAL_A8_C9_EXPECTED_CONDITIONAL,MAC(G)+MAC(B_phi)+p_activate*(MAC(R_psi)+MAC(S...,3.243790e+10,True,True,True,ANALYTICAL_MAC_LOWER_BOUND
3,GATED_EXECUTE_ALL_C9,MAC(G)+MAC(B_phi)+MAC(R_psi)+MAC(S_omega),3.478244e+10,True,True,True,ANALYTICAL_MAC_LOWER_BOUND
4,RATIO_A8C9_vs_C1,E/C1,1.028086e+00,NaN,NaN,NaN,ANALYTICAL_RATIO
5,RATIO_A8C9_vs_literal_C2,E/C2,9.325952e-01,NaN,NaN,NaN,ANALYTICAL_RATIO
6,SAVING_vs_literal_C2,C2-E,2.344502e+09,NaN,NaN,NaN,ANALYTICAL_SAVING_MACS
7,RATIO_A8C9_vs_gated_C9,E/gated,9.325942e-01,NaN,NaN,NaN,ANALYTICAL_RATIO
8,SAVING_vs_gated_C9,gated-E,2.344539e+09,NaN,NaN,NaN,ANALYTICAL_SAVING_MACS


## 4. Batch-1 latency (synthetic hardware microbenchmark)

In [5]:
tshow("phase34_batch1_latency_results_v1_3.csv")

phase34_batch1_latency_results_v1_3.csv | shape: (6, 21)
                    arm                       scope  batch_size  warmup_count  measured_iterations                                        task_schedule     mean_ms  std_ms      p50_ms      p90_ms      p95_ms      p99_ms      min_ms      max_ms                                                      timer_type  wall_clock_cross_check_ms                                                            precision_semantics                                                  hardware        alias_names  alias_measured_mean_ms  independent_evidence
         GLOBAL_ONLY_C1  MODEL_COMPUTE_PATH_RUNTIME           1            50                 1000 iteration_index mod 4 (0=ISIC,1=BUSI,2=KVASIR,3=KUB) 3120.524316     NaN 3124.829102 3135.845459 3135.845459 3135.845459 3098.464355 3135.845459 torch.cuda.Event(enable_timing=True) + torch.cuda.synchronize()                 3125.31724 G canonical: float32 + autocast(fp16) + torch.no_grad() + inference_mod

,arm,scope,batch_size,warmup_count,measured_iterations,task_schedule,mean_ms,std_ms,p50_ms,p90_ms,...,p99_ms,min_ms,max_ms,timer_type,wall_clock_cross_check_ms,precision_semantics,hardware,alias_names,alias_measured_mean_ms,independent_evidence
0,GLOBAL_ONLY_C1,MODEL_COMPUTE_PATH_RUNTIME,1,50,1000,"iteration_index mod 4 (0=ISIC,1=BUSI,2=KVASIR,...",3120.524316,NaN,3124.829102,3135.845459,...,3135.845459,3098.464355,3135.845459,torch.cuda.Event(enable_timing=True) + torch.c...,3125.31724,G canonical: float32 + autocast(fp16) + torch....,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,...",G_COMPONENT,3132.053125,True
1,R_PSI_COMPONENT,SYNTHETIC_COMPONENT_RUNTIME,1,50,1000,"iteration_index mod 4 (0=ISIC,1=BUSI,2=KVASIR,...",937.517871,NaN,936.504333,957.986755,...,957.986755,918.030334,957.986755,torch.cuda.Event(enable_timing=True) + torch.c...,937.57162,float32 + autocast disabled + torch.inference_...,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,...",NaN,NaN,True
2,B_PHI_COMPONENT,SYNTHETIC_COMPONENT_RUNTIME,1,50,1000,"iteration_index mod 4 (0=ISIC,1=BUSI,2=KVASIR,...",76.909752,NaN,74.501122,82.506752,...,82.506752,72.742813,82.506752,torch.cuda.Event(enable_timing=True) + torch.c...,76.96760,float32 + autocast disabled + torch.inference_...,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,...",NaN,NaN,True
3,S_OMEGA_COMPONENT,SYNTHETIC_COMPONENT_RUNTIME,1,50,1000,"iteration_index mod 4 (0=ISIC,1=BUSI,2=KVASIR,...",45.097177,NaN,45.156418,45.975552,...,45.975552,44.290047,45.975552,torch.cuda.Event(enable_timing=True) + torch.c...,45.15324,float32 + autocast disabled + torch.inference_...,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,...",NaN,NaN,True
4,SKIP_CONTROLLED_PATH,MODEL_COMPUTE_PATH_RUNTIME,1,50,1000,"iteration_index mod 4 (0=ISIC,1=BUSI,2=KVASIR,...",3143.793652,NaN,3140.082764,3161.132080,...,3161.132080,3128.846436,3161.132080,torch.cuda.Event(enable_timing=True) + torch.c...,3143.79570,G canonical + B_phi canonical,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,...",NaN,NaN,True
5,EXECUTE_CONTROLLED_PATH,MODEL_COMPUTE_PATH_RUNTIME,1,50,1000,"iteration_index mod 4 (0=ISIC,1=BUSI,2=KVASIR,...",3740.713281,NaN,3743.125488,3748.511719,...,3748.511719,3731.411865,3748.511719,torch.cuda.Event(enable_timing=True) + torch.c...,3740.66044,G canonical + B_phi/R_psi/S_omega canonical,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,...",FINAL_ROUTED_A8_C9,3734.817334,True


## 5. Batch-8 throughput

In [6]:
tshow("phase34_batch8_throughput_results_v1_3.csv")

phase34_batch8_throughput_results_v1_3.csv | shape: (6, 10)
                    arm  batch_size  measured_batches  processed_samples  synchronized_seconds  samples_per_second                                fps_definition          precision_semantics                timing_scope                                                  hardware
         GLOBAL_ONLY_C1           8              1000               8000             85.414416           93.661005 processed_samples / synchronized_wall_seconds component-specific canonical  MODEL_COMPUTE_PATH_RUNTIME NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,232 B VRAM)
        R_PSI_COMPONENT           8              1000               8000             13.321764          600.521068 processed_samples / synchronized_wall_seconds component-specific canonical SYNTHETIC_COMPONENT_RUNTIME NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,232 B VRAM)
        B_PHI_COMPONENT           8              1000               8000              0.386081        20721.0457

,arm,batch_size,measured_batches,processed_samples,synchronized_seconds,samples_per_second,fps_definition,precision_semantics,timing_scope,hardware
0,GLOBAL_ONLY_C1,8,1000,8000,85.414416,93.661005,processed_samples / synchronized_wall_seconds,component-specific canonical,MODEL_COMPUTE_PATH_RUNTIME,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,..."
1,R_PSI_COMPONENT,8,1000,8000,13.321764,600.521068,processed_samples / synchronized_wall_seconds,component-specific canonical,SYNTHETIC_COMPONENT_RUNTIME,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,..."
2,B_PHI_COMPONENT,8,1000,8000,0.386081,20721.045771,processed_samples / synchronized_wall_seconds,component-specific canonical,SYNTHETIC_COMPONENT_RUNTIME,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,..."
3,S_OMEGA_COMPONENT,8,1000,8000,0.231665,34532.622537,processed_samples / synchronized_wall_seconds,component-specific canonical,SYNTHETIC_COMPONENT_RUNTIME,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,..."
4,SKIP_CONTROLLED_PATH,8,1000,8000,85.859384,93.175604,processed_samples / synchronized_wall_seconds,component-specific canonical,MODEL_COMPUTE_PATH_RUNTIME,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,..."
5,EXECUTE_CONTROLLED_PATH,8,1000,8000,99.672479,80.262878,processed_samples / synchronized_wall_seconds,component-specific canonical,MODEL_COMPUTE_PATH_RUNTIME,"NVIDIA GeForce RTX 3060 Ti (cc 8.6, 8,589,279,..."


## 6. GPU memory

In [7]:
tshow("phase34_gpu_memory_results_v1_3.csv")

phase34_gpu_memory_results_v1_3.csv | shape: (6, 8)
                    arm  checkpoint_file_bytes  parameter_bytes  baseline_gpu_allocated_after_model_load  baseline_gpu_reserved_after_model_load  peak_gpu_allocated_during_inference  peak_gpu_reserved_during_inference  incremental_peak_allocated_over_loaded_baseline
         GLOBAL_ONLY_C1               42369710         14092004                                 26690560                                48234496                            336083968                           411041792                                        309393408
        R_PSI_COMPONENT               15425424          5105604                                  9311744                                10485760                             48668160                            73400320                                         39356416
        B_PHI_COMPONENT                 455682           147204                                   155136                                 2097152   

,arm,checkpoint_file_bytes,parameter_bytes,baseline_gpu_allocated_after_model_load,baseline_gpu_reserved_after_model_load,peak_gpu_allocated_during_inference,peak_gpu_reserved_during_inference,incremental_peak_allocated_over_loaded_baseline
0,GLOBAL_ONLY_C1,42369710,14092004,26690560,48234496,336083968,411041792,309393408
1,R_PSI_COMPONENT,15425424,5105604,9311744,10485760,48668160,73400320,39356416
2,B_PHI_COMPONENT,455682,147204,155136,2097152,9725952,23068672,9570816
3,S_OMEGA_COMPONENT,7816,4868,8704,2097152,9577984,23068672,9569280
4,SKIP_CONTROLLED_PATH,42825392,14239208,26845696,48234496,336239104,411041792,309393408
5,EXECUTE_CONTROLLED_PATH,58258632,19349680,36166144,56623104,345559552,423624704,309393408


## 7. Controlled SKIP / EXECUTE path runtime

In [8]:
lat = pd.read_csv(TAB / "phase34_batch1_latency_results_v1_3.csv")
for arm in ("SKIP_CONTROLLED_PATH", "EXECUTE_CONTROLLED_PATH"):
    r = lat[lat["arm"] == arm].iloc[0]
    print(f"  {arm:24s} mean={r['mean_ms']:.6f} ms  p50={r['p50_ms']:.6f} "
          f"scope={r['scope']}")


  SKIP_CONTROLLED_PATH     mean=3143.793652 ms  p50=3140.082764 scope=MODEL_COMPUTE_PATH_RUNTIME
  EXECUTE_CONTROLLED_PATH  mean=3740.713281 ms  p50=3743.125488 scope=MODEL_COMPUTE_PATH_RUNTIME


## 8. Conditional weighted runtime estimate

In [9]:
tshow("phase34_conditional_runtime_estimate_v1_3.csv")

phase34_conditional_runtime_estimate_v1_3.csv | shape: (5, 4)
      quantity       value     unit                                                                                                                                                                                                       note
   L_SKIP_mean 3143.793652       ms                                                                                                                                                                                           SKIP = G + B_phi
L_EXECUTE_mean 3740.713281       ms                                                                                                                                                                      EXECUTE = G + B_phi + R_psi + S_omega
        p_SKIP    0.725714 fraction                                                                                                                                                                                          

,quantity,value,unit,note
0,L_SKIP_mean,3143.793652,ms,SKIP = G + B_phi
1,L_EXECUTE_mean,3740.713281,ms,EXECUTE = G + B_phi + R_psi + S_omega
2,p_SKIP,0.725714,fraction,635/875
3,p_EXECUTE,0.274286,fraction,240/875 (ETA 236 + ETR 4)
4,L_weighted,3307.520179,ms,WEIGHTED_HARDWARE_SPECIFIC_RUNTIME_ESTIMATE; N...


## 9. Claim limitations and protected-access state

In [10]:
res = jload(AUD / "phase34_efficiency_result_v1_3.json")
print("claims not made:", res["claims_not_made"])
print()
print("scope limitations:", res["scope_limitations"])
print()
prov = jload(AUD / "phase34_runtime_execution_provenance_v1_3.json")
print("protected-data counters:")
for k, v in prov["protected_data_counters"].items():
    print(f"  {k:28s} = {v}")
print()
print("aliases (measured once, not double-counted):", prov["aliases"])
print("distinct measured arms:", prov["distinct_measured_arms"])
print("pre-timing smoke forwards during measured run:", prov["premeasurement_smoke_forwards"])


claims not made: ['real-time capability', 'mobile suitability', 'energy efficiency', 'hardware-independent speed', 'clinical deployment readiness', 'causal efficiency superiority']

scope limitations: timed region covers module forwards (and sigmoid where frozen); preprocessing, host-device transfer, ROI generation, crop/resizing, fusion and decision logic are excluded per the frozen timed-operation registry

protected-data counters:
  raw_image_reads              = 0
  raw_gt_reads                 = 0
  router_holdout_raw_reads     = 0
  test_reads                   = 0
  training_steps               = 0

aliases (measured once, not double-counted): {'G_COMPONENT': 'GLOBAL_ONLY_C1', 'FINAL_ROUTED_A8_C9': 'EXECUTE_CONTROLLED_PATH'}
distinct measured arms: ['GLOBAL_ONLY_C1', 'R_PSI_COMPONENT', 'B_PHI_COMPONENT', 'S_OMEGA_COMPONENT', 'SKIP_CONTROLLED_PATH', 'EXECUTE_CONTROLLED_PATH']
pre-timing smoke forwards during measured run: 10


## 10. Execution audit and final check

In [11]:
au = jload(AUD / "phase34_efficiency_execution_audit_v1_3.json")
print("audit all_pass:", au["all_pass"], "| harness:", au["frozen_harness_used"],
      "| worker:", au["frozen_worker_used"])
fc = jload(AUD / "phase34_execution_final_check_v1_3.json")
print("final check:", fc["n_checks"], "checks,", fc["n_failed"], "failed")
term = jload(AUD / "phase34_execution_terminal_v1_3.json")
print()
print("terminal:")
print(" ", term["terminal"])


audit all_pass: True | harness: True | worker: True
final check: 41 checks, 0 failed

terminal:
  PHASE34_EFFICIENCY_RUNTIME_PROFILING_V1_3_COMPLETE__FROZEN_V1_3_HARNESS_AND_CANONICAL_COMPONENT_SEMANTICS_USED__BATCH1_SYNTHETIC_COMPONENT_AND_CONTROLLED_PATH_LATENCY_BATCH8_THROUGHPUT_GPU_MEMORY_STATIC_RESOURCES_AND_CONDITIONAL_WEIGHTED_RUNTIME_PROFILED__ANALYTICAL_MAC_EVIDENCE_PRESERVED_SEPARATELY__NO_PROTECTED_RAW_OR_TEST_ACCESS__PHASE34_CLOSURE_ELIGIBLE_NOT_STARTED


---
## Summary

| evidence class | content |
|---|---|
| HISTORICAL_ANALYTICAL_COMPUTE | C1 31,551,740,928 · C2 34,782,403,584 · A8/C9 32,437,902,022.58 · gated C9 34,782,440,912 MACs |
| STATIC_MODEL_RESOURCE_PROFILE | G 3,523,001 · R_psi 1,276,401 · B_phi 36,801 · S_omega 1,217 (total 4,837,420) |
| MEASURED_HARDWARE_SPECIFIC_RUNTIME_PROFILE | batch-1 latency + batch-8 throughput + GPU memory on RTX 3060 Ti |
| CONDITIONAL_COMPUTATION_PROFILE | weighted runtime from frozen Phase30 states (qualified) |

Scope labels: `MODEL_COMPUTE_PATH_RUNTIME` / `SYNTHETIC_COMPONENT_RUNTIME`.
`END_TO_END_PIPELINE_RUNTIME` **not** claimed. TEST locked · Phase35 not started.
